# Fix Label Tambang Papua — re-export label (union +Maus) untuk 2 tile terdampak + patch in-place

**Konteks.** `qc_label_tambang_papua.ipynb` menemukan: label Tambang test/train sekarang murni
Tang & Werner 2023 (patch sudah dipotong sebelum union +Maus 2022 ditambahkan ke kode), gap
+38,7% area. Tile Papua yang overlap poligon Maus HANYA **2 dari 36**: `papua_t2_tile_21`
(area Grasberg/Tembagapura) dan `papua_t2_tile_32` (tenggara).

**Kenapa tidak generate ulang `split_files()`?** Val/test TIDAK punya manifest per-file —
assignment-nya cuma fisik (file sudah masuk `val.tar`/`test.tar` di `Bahan_Training_Fix`).
`split_files()` pakai `rng.permutation(seed=42)` atas urutan list file SAAT itu — kalau
dijalankan ulang dengan jumlah/urutan file yang sedikit berbeda, peta index->file bisa BERGESER
TOTAL untuk file-file LAIN juga (bukan cuma yang baru). Itu bisa diam-diam mengubah identitas
test-set yang sudah jadi acuan `metrics.json` — risikonya kebocoran/pergeseran split.

**Solusi aman (dipakai di sini): patch `lab` IN-PLACE.** Tiap `.npz` di dalam tar menyimpan
`tile`/`row`/`col` (lihat `cut_patches()` di `patches.py`). Untuk patch yang `tile`-nya salah
satu dari 2 tile terdampak: hitung ulang `lab` dari label GEE yang sudah di-refresh, di window
piksel `row:row+256, col:col+256` yang SAMA. `img` dan lokasi split (train/val/test) TIDAK
berubah sama sekali.

**Desain aman:**
- Hasil surgery ditulis ke folder **BARU** (`Bahan_Training_Fix_LabelFix/`), TIDAK menimpa
  `Bahan_Training_Fix` asli — supaya data yang sudah dipakai utk `metrics.json` model 1/2/3
  sekarang tidak tersentuh sampai kamu yakin hasilnya benar.
- `DRY_RUN=True` default di cell surgery — cuma melaporkan berapa patch akan berubah,
  TIDAK menulis apa pun, sampai kamu sengaja set `False`.
- Re-export GEE HANYA band `label` (bukan citra Sentinel-2 yang tidak berubah) untuk 2 tile
  saja — ringan & cepat dibanding re-export penuh.

In [ ]:
# === SETUP (Colab) — clone repo + install + mount Drive ===
import sys, subprocess, importlib
from pathlib import Path

subprocess.run(
    "cd /content && (git -C fw_repo pull -q || git clone --depth 1 "
    "https://github.com/Ridho-Dwi-Syahputra/forestwatch-model.git fw_repo)",
    shell=True, check=False,
)
subprocess.run("pip install -q -e /content/fw_repo[gee,gis,ml]", shell=True, check=False)
if "/content/fw_repo/src" not in sys.path:
    sys.path.insert(0, "/content/fw_repo/src")
for _m in [m for m in list(sys.modules) if m == "forestwatch" or m.startswith("forestwatch.")]:
    del sys.modules[_m]
importlib.invalidate_caches()

from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/Satria Data 3.0")
print("Setup selesai. DRIVE_ROOT:", DRIVE_ROOT)

In [ ]:
# === AUTH GEE + KONFIGURASI + 2 TILE TERDAMPAK (dari qc_label_tambang_papua.ipynb) ===
import ee
from forestwatch.gee.auth import init_ee
from forestwatch.constants import PAPUA_BBOX
from forestwatch.config import load_config
from forestwatch.gee.tiles import make_tiles_from_bbox

cfg = load_config()
init_ee(project=cfg["project"]["gee_project_id"])
T2 = cfg["periods"]["t2"]

NX, NY = cfg["export"]["tiles_nx"], cfg["export"]["tiles_ny"]
tile_bboxes = make_tiles_from_bbox(PAPUA_BBOX, nx=NX, ny=NY)

# Hasil qc_label_tambang_papua.ipynb: index 21 & 32 overlap poligon Maus.
AFFECTED_IDX = [21, 32]
for idx in AFFECTED_IDX:
    print(f"papua_t2_tile_{idx:02d}.tif  bbox={tile_bboxes[idx]}")

In [ ]:
# === RE-EXPORT LABEL SAJA (union +Maus, kode build_label() sekarang) utk 2 tile ===
from forestwatch.gee.label_fusion import build_label
from forestwatch.gee.export import export_stack

lf = cfg["label_fusion"]
LABEL_FIX_FOLDER = "Label_Fix_Tambang_Maus"  # subfolder baru di Drive root

tasks = []
for idx in AFFECTED_IDX:
    xmin, ymin, xmax, ymax = tile_bboxes[idx]
    tile_geom = ee.Geometry.Rectangle([xmin, ymin, xmax, ymax])
    label_new = build_label(
        tile_geom, T2,
        hansen_loss_year_min=lf["hansen_loss_year_min"],
        hansen_erosion_pixels=lf["hansen_erosion_pixels"],
        palm_prob_threshold=lf["palm_prob_threshold"],
        esa_forest_union=lf.get("esa_forest_union", False),
    )  # build_label sudah .toByte().clip(region) -- sama persis spt Bagian 7 full_pipeline
    desc = f"papua_t2_tile_{idx:02d}_label_fix"
    task = export_stack(
        label_new, description=desc, folder=LABEL_FIX_FOLDER,
        region=tile_geom, scale=cfg["sentinel2"]["scale"],
    )
    tasks.append((desc, task))

print("Submitted", len(tasks), "task export ke folder Drive:", LABEL_FIX_FOLDER)
for desc, _ in tasks:
    print(" -", desc)
print()
print("Pantau di https://code.earthengine.google.com/tasks -- TUNGGU status COMPLETED")
print("keduanya sebelum lanjut ke cell berikutnya (band tunggal, 2 tile -- harusnya cepat).")

In [ ]:
# === VERIFIKASI: GeoTIFF label baru sudah ada di Drive (jalankan SETELAH task COMPLETED) ===
# PENTING: ee.batch.Export.image.toDrive(folder=...) SELALU bikin folder di ROOT 'My Drive',
# bukan nested di DRIVE_ROOT ('Satria Data 3.0') -- sama spt semua ekspor lain di proyek ini
# (folder=TILES_T2.name dst juga begitu). Kalau 2 folder 'Label_Fix_Tambang_Maus' terpisah
# muncul di root Drive (race condition GEE saat 2 task jalan bersamaan), GABUNGKAN manual
# dulu jadi 1 folder isi 2 file sebelum jalankan cell ini.
MY_DRIVE_ROOT = Path("/content/drive/MyDrive")
LABEL_FIX_DIR = MY_DRIVE_ROOT / LABEL_FIX_FOLDER
AFFECTED_TILE_FILES = {}
for idx in AFFECTED_IDX:
    tile_name = f"papua_t2_tile_{idx:02d}.tif"  # nama sesuai field 'tile' di .npz (basename ubin asal)
    fix_path = LABEL_FIX_DIR / f"papua_t2_tile_{idx:02d}_label_fix.tif"
    ok = fix_path.exists()
    status = "OK" if ok else "BELUM ADA (cek GEE Tasks / gabungkan folder duplikat di Drive root)"
    print(f"{tile_name:<28} -> {fix_path.name}  {status}")
    if ok:
        AFFECTED_TILE_FILES[tile_name] = fix_path

assert len(AFFECTED_TILE_FILES) == len(AFFECTED_IDX), "Tunggu semua export task COMPLETED dulu."


In [ ]:
# === SURGERY: patch 'lab' IN-PLACE di tar Bahan_Training_Fix, output ke folder BARU ===
# DRY_RUN=True (default): cuma laporan, TIDAK menulis apa pun ke Drive.
# Set False HANYA setelah preview di bawah terlihat wajar (jumlah patch & split masuk akal).
DRY_RUN = True

import io, tarfile
import numpy as np
import rasterio
from tqdm.auto import tqdm

PATCH_SIZE = cfg["patches"]["size"]
BAHAN_DIR = DRIVE_ROOT / "Bahan_Training_Fix"
OUT_DIR = DRIVE_ROOT / "Bahan_Training_Fix_LabelFix"  # folder BARU, asli tak tersentuh

new_label_rasters = {}
for tile_name, path in AFFECTED_TILE_FILES.items():
    with rasterio.open(path) as src:
        new_label_rasters[tile_name] = src.read(1).astype("uint8")
    print("Label baru dimuat:", tile_name, "shape =", new_label_rasters[tile_name].shape)


def patch_npz_bytes(raw_bytes):
    """Return (bytes_baru, changed). lab diganti kalau 'tile' termasuk yang terdampak."""
    data = np.load(io.BytesIO(raw_bytes))
    tile = str(data["tile"])
    if tile not in new_label_rasters:
        return raw_bytes, False
    row, col = int(data["row"]), int(data["col"])
    full = new_label_rasters[tile]
    new_lab = full[row:row + PATCH_SIZE, col:col + PATCH_SIZE]
    if new_lab.shape != (PATCH_SIZE, PATCH_SIZE):
        return raw_bytes, False  # di luar batas raster baru -- aman, skip
    out = io.BytesIO()
    np.savez(out, img=data["img"], lab=new_lab, tile=data["tile"], row=data["row"], col=data["col"])
    return out.getvalue(), True


def process_tar(tar_path, out_path):
    changed, total = 0, 0
    if not DRY_RUN:
        out_path.parent.mkdir(parents=True, exist_ok=True)
    mode_out = "w" if not DRY_RUN else None
    tout = tarfile.open(out_path.with_suffix(".tar.partial"), mode_out) if not DRY_RUN else None
    with tarfile.open(tar_path, "r") as tin:
        for member in tin.getmembers():
            total += 1
            raw = tin.extractfile(member).read()
            new_raw, did_change = patch_npz_bytes(raw)
            if did_change:
                changed += 1
            if not DRY_RUN:
                info = tarfile.TarInfo(name=member.name)
                info.size = len(new_raw)
                info.mtime = member.mtime
                tout.addfile(info, io.BytesIO(new_raw))
    if not DRY_RUN:
        tout.close()
        out_path.with_suffix(".tar.partial").rename(out_path)  # atomic
    return changed, total


tar_files = []
for split in ["train", "val", "test"]:
    split_dir = BAHAN_DIR / split
    if split_dir.exists():
        tar_files += [(split, p) for p in sorted(split_dir.glob("*.tar"))]

print(f"{'[DRY RUN] ' if DRY_RUN else ''}Memproses {len(tar_files)} tar dari {BAHAN_DIR}...")
summary = {}
for split, tp in tqdm(tar_files, desc="Tar"):
    out_p = OUT_DIR / split / tp.name
    changed, total = process_tar(tp, out_p)
    summary.setdefault(split, [0, 0])
    summary[split][0] += changed
    summary[split][1] += total
    print(f"  {split}/{tp.name}: {changed}/{total} patch terdampak")

print()
print("RINGKASAN per split:")
for split, (changed, total) in summary.items():
    print(f"  {split}: {changed} patch diganti labelnya (dari {total} total)")
if DRY_RUN:
    print()
    print("Ini DRY RUN -- tidak ada file ditulis. Kalau ringkasan di atas wajar, set")
    print("DRY_RUN=False lalu jalankan ulang cell ini utk eksekusi nyata ke", OUT_DIR)

## Setelah `DRY_RUN=False` berhasil

1. `Bahan_Training_Fix_LabelFix/{train,val,test}/*.tar` berisi salinan PENUH dataset dengan label
   Tambang ter-refresh untuk 2 tile terdampak -- `Bahan_Training_Fix` asli TIDAK berubah.
2. Copy juga `train_rajaampat/`, `patch_sampler_weights_shared.json`, `class_weights.json` dari
   `Bahan_Training_Fix` asli ke `Bahan_Training_Fix_LabelFix` (tidak terdampak, tapi training
   notebook butuh semuanya ada di folder yang sama).
3. Hitung ulang `class_weights.json` & `patch_sampler_weights_shared.json` HANYA bila perlu --
   perubahan 2 tile dari 36 kemungkinan tidak signifikan mengubah distribusi kelas global, tapi
   cek dulu di `optimize_dataset.ipynb` sebelum asumsi aman dilewati.
4. Setelah yakin, baru putuskan: ganti nama folder (`Bahan_Training_Fix` -> `_old`, lalu
   `Bahan_Training_Fix_LabelFix` -> `Bahan_Training_Fix`) supaya notebook training existing
   otomatis pakai data baru tanpa ubah path di notebook training.
5. Re-evaluasi `metrics.json` test set model 1 dgn data yang sudah di-refresh (tanpa training
   ulang -- checkpoint sama, cuma ganti `test_loader` baca dari folder baru) untuk lihat
   seberapa besar Tambang IoU naik MURNI dari perbaikan label, sebelum fine-tune loss-function.